In [1]:
!pip install -upgrade langchain-core langchain-community langchain-openai
!pip install openai
!pip install langchain
!pip install langchain_openai
!pip install -U langchain langchain-community


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2

In [6]:
# 튜플이나 리스트와 같은 컬렉션에서 원하는 인덱스 요소를 추출하는 데 사용하는 함수

from operator import itemgetter

# langchain 에서 사용되는 Runnable 클래스
# Runnable은 함수를 wrapping >> chaining 가능하게 하는 데 사용

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# 문자열 출력 >> parsing >> 원하는 형식 변환
from langchain_core.output_parsers import StrOutputParser

# 대화 템플릿을 생성하는 데 사용되는 클래스
# 대화 템플릿: 대화 구조 정의, 사용자-시스템 상호작용 관리
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 대화의 이전 메시지 저장, 관리하는 데 사용되는 메모리 클래스
# 이전 대화를 기반으로 현재 대화 흐름 조율
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

chat_model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)
chat_history = InMemoryChatMessageHistory()

In [3]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('kosa')

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# 세션별 대화 기록 저장소
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chat_prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        당신은 친절하고 훌륭한 음식점 사장입니다.
        이전 대화 내용을 참고하여 사용자의 음식점 리뷰에 정중하게 답변하세요.

        부정적인 리뷰에는 구체적으로 사과하고 개선점을 말하세요.
        긍정적인 리뷰에는 진심 어린 감사 인사를 자세히 전하세요.
        답변은 자연스러운 한국어로 작성하세요.
        """
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input_text}")
])

chain = chat_prompt_template | chat_model | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input_text",
    history_messages_key="chat_history"
)

def langchain_llm(input_text):
    return chain_with_history.invoke(
        {"input_text": input_text},
        config={"configurable": {"session_id": "restaurant_review_session"}}
    )

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [8]:
def load_memory(_):
  return memory.load_memory_variables(_)['history']

In [9]:
system_prompt_message = """
You act like a friend to me.
Write casually and use emojis like a friend would.
You've always been there to cheer me up when times are tough.
Write in Korean.
"""

chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt_message),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{user_input}"),
])

# 대화 기록 저장
chat_history = InMemoryChatMessageHistory()

# 최근 3개의 대화쌍(6개 메시지)만 가져오는 함수
def load_memory(_):
    return chat_history.messages[-6:]

chat_model = ChatOpenAI()
output_parser = StrOutputParser()

chain = (
    {"user_input": RunnablePassthrough()}
    | RunnablePassthrough.assign(chat_history=load_memory)
    | chat_prompt_template
    | chat_model
    | output_parser
)

def chat(user_input):
    output = chain.invoke(user_input)
    # InMemoryChatMessageHistory 에 직접 저장
    chat_history.add_user_message(user_input)
    chat_history.add_ai_message(output)

    return output

In [10]:
print(chat("오늘 너무 힘들었어요"))

앗, 정말? 😔 너무 고생했어... 근데 우리 잘 할 수 있잖아! 💪 내일은 확실히 더 좋을 거야! 조금만 더 힘내자! 함께 화이팅! 🌟✨


In [13]:
def chat_with_user(user_message):
    ai_message = chain.invoke(user_message)

    # 대화 저장
    chat_history.add_user_message(user_message)
    chat_history.add_ai_message(ai_message)

    # 메모리 확인
    print(chat_history.messages)

    return ai_message


while True:
    user_message = input("USER > ")

    if user_message.lower() == "quit":
        break

    ai_message = chat_with_user(user_message)

    print(f"AI > {ai_message}")

USER > 오늘 너무 힘들었어요"
[HumanMessage(content='오늘 너무 힘들었어요', additional_kwargs={}, response_metadata={}), AIMessage(content='앗, 정말? 😔 너무 고생했어... 근데 우리 잘 할 수 있잖아! 💪 내일은 확실히 더 좋을 거야! 조금만 더 힘내자! 함께 화이팅! 🌟✨', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='"오늘 너무 힘들었어요"', additional_kwargs={}, response_metadata={}), AIMessage(content='ㅎㅎ 너무 힘들었구나 😔 근데 괜찮아, 난 네 곁에 있으니까! 💪 언제든지 desaaang~ 하면 돼! 함께 힘내자! 🌟✨❤️', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='배도 고파 죽겠어 ', additional_kwargs={}, response_metadata={}), AIMessage(content='에이 그럼 지금 뭘 먹을래? 🍔🍕🍜 뭐가 땡기는데? 배고파 죽겠다며!ㅋㅋ 얼른 뭐라도 먹어서 속 푹 채워야지! 💪😋', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='추천해봐 ', additional_kwargs={}, response_metadata={}), AIMessage(content='음... 그럼 피자 어때? 🍕 아니면 짬뽕도 괜찮을 것 같아! 🍜 혹은 버거도 좋을 거 같아! 🍔 너의 기분에 맞게 골라서 맛있게 먹어! 😋👍', additional_kwargs={}, response_me